In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
!pip install -q pyngrok fastapi uvicorn pydantic

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(
    torch.cuda.memory_allocated(0) / 1024**3, 2
), "GB")

In [ ]:
import json
import threading
import uvicorn
import torch

from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# 1. Read secrets
user_secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
MODEL_API_KEY = user_secrets.get_secret("MODEL_API_KEY")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 2. Create app
app = FastAPI(title="Contract Analysis LLM API - Qwen3-8B")

class ContractRequest(BaseModel):
    contract_text: str
    language: str = "en"

class PromptRequest(BaseModel):
    prompt: str

# 3. Shared generation helper
def generate_text(messages, max_new_tokens):
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

# 4. Health endpoint
@app.get("/health")
def health():
    return {"status": "ok", "model": "Qwen3-8B"}

# 5. Original analyze endpoint
@app.post("/analyze")
def analyze_contract(
    request: ContractRequest,
    x_api_key: str = Header(default="")
):
    if x_api_key != MODEL_API_KEY:
        raise HTTPException(status_code=401, detail="Unauthorized")

    if not request.contract_text.strip():
        raise HTTPException(status_code=400, detail="contract_text cannot be empty")

    messages = [
        {
            "role": "system",
            "content": (
                "You analyze contract clauses and explain potential risks. "
                "Return only valid JSON with keys: contract_type, summary, "
                "risks, advice. Each risk must contain clause, reason, "
                "severity, legal_reference. Severity must be low, medium, "
                "or high. Do not invent legal references. "
                "This is preliminary analysis, not verified legal advice."
            )
        },
        {
            "role": "user",
            "content": (
                f"Language: {request.language}\n"
                f"Analyze this contract:\n{request.contract_text}"
            )
        }
    ]

    answer = generate_text(messages, max_new_tokens=500)

    try:
        return json.loads(answer)
    except json.JSONDecodeError:
        raise HTTPException(
            status_code=502,
            detail="The model did not return valid JSON"
        )

# 6. Benchmark endpoint
@app.post("/generate")
def generate_from_prompt(
    request: PromptRequest,
    x_api_key: str = Header(default="")
):
    if x_api_key != MODEL_API_KEY:
        raise HTTPException(status_code=401, detail="Unauthorized")

    if not request.prompt.strip():
        raise HTTPException(status_code=400, detail="prompt cannot be empty")

    answer = generate_text(
        [{"role": "user", "content": request.prompt}],
        max_new_tokens=1200
    )

    if not answer:
        raise HTTPException(status_code=502, detail="The model returned an empty response")

    return {"text": answer}

# 7. Start server and ngrok
ngrok.kill()
tunnel = ngrok.connect(8000, "http")

print("Model: Qwen3-8B")
print("Public URL:", tunnel.public_url)
print("Health endpoint:", tunnel.public_url + "/health")
print("Generate endpoint:", tunnel.public_url + "/generate")
print("Keep this Kaggle session running.")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

In [ ]:
import requests
from kaggle_secrets import UserSecretsClient

# 1. Paste the Public URL printed by your API cell
API_URL = "https://important-docile-unmindful.ngrok-free.dev"

# 2. Load the API key from Kaggle Secrets
user_secrets = UserSecretsClient()
API_KEY = user_secrets.get_secret("MODEL_API_KEY")

# 3. Test the health endpoint
print("Testing API health...")

try:
    health_response = requests.get(
        f"{API_URL}/health",
        timeout=20
    )

    print("Health status:", health_response.status_code)
    print("Health response:", health_response.text)

    if health_response.status_code != 200:
        print("Health test failed. Check the Public URL and API server.")
    else:
        # 4. Test model generation
        print("\nTesting Qwen3-8B generation...")

        test_prompt = """
Analyze this contract clause:
"The company may change any fees at any time without notice."

Return JSON with a risk description and severity.
"""

        response = requests.post(
            f"{API_URL}/generate",
            headers={"x-api-key": API_KEY},
            json={"prompt": test_prompt},
            timeout=180
        )

        print("Generation status:", response.status_code)
        print("Model response:")
        print(response.text)

except requests.exceptions.Timeout:
    print("Request timed out. Check whether the Kaggle API cell is still running.")
except requests.exceptions.RequestException as e:
    print("Connection error:", str(e))